# TabM on the funding panel, and what a network adds that a tree does not

[`06_linear`](06_linear.ipynb) and [`07_gbm`](07_gbm.ipynb) read the same design matrix this
notebook does: one row per perpetual per settlement, one column per feature, with nothing in
the table saying the rows are ordered in time. They differ in what they can represent. A
penalized linear model gives each feature one coefficient and can spread weight across a group
of near-duplicate columns. A tree ensemble can express an interaction - a condition on one
feature evaluated inside a region defined by others - but it reaches one by choosing a single
column at each split, and several columns here carry almost the same information, so which one
is chosen is close to arbitrary.

A neural network on the same table answers the same question a third way. Its first layer is a
weighted sum of every feature, so like the linear model it never has to choose among correlated
columns; the nonlinearity after it lets those sums combine into interactions the linear model
cannot write down. That is the reason to fit one here, rather than a general preference for
neural networks: the two properties that pulled against each other in the previous two
notebooks are not obviously in conflict in this architecture.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 資金費率面板上的 TabM：神經網路能補上樹模型沒有的什麼

[`06_linear`](06_linear.ipynb) 與 [`07_gbm`](07_gbm.ipynb) 讀取的設計矩陣與本 Notebook 相同：每份永續合約每次結算一列，每項特徵一欄，表格本身並未標示資料列的時間順序。兩者差在可表示的關係。帶懲罰的線性模型為每項特徵賦予一個係數，可在近乎重複的欄位群組中分散權重。樹集成模型能表示交互作用，也就是在其他特徵界定的區域內檢查某項特徵的條件，但每次分割都須選擇單一欄位；此處數個欄位含有幾乎相同資訊，因此選到哪個欄位近乎任意。

對同一表格使用神經網路，是第三種回答相同問題的方式。第一層對所有特徵做加權和，因此像線性模型一樣，不必在相關欄位之間擇一；後續非線性轉換讓這些加權和組成線性模型無法表示的交互作用。此處擬合神經網路的理由在此，而非一般性偏好神經網路：前兩份 Notebook 中彼此牽制的兩項性質，在這種架構下未必衝突。
<!-- ML4T_ZH_END c0000-p000 -->

**TabM is an ensemble, and the ensemble is the point.** Averaging several independently
initialized networks is a standard way to make a neural fit on a table less erratic, and the
cost is normally that you train several networks. TabM trains most of one. A backbone of two
layers is shared by every member; each member owns only a vector carrying one number per hidden
unit, which scales the backbone's output element by element, and its own final linear layer.
The members' predictions are averaged. So `n_members: 4` at `hidden_dim: 64` costs four small
vectors and four output layers on top of one backbone, not four networks - which is why the
member count can be raised much further than the width can.

**This notebook fits three of the four declared labels, and two of them are not returns.**
`fwd_ret_8h` is a regression target. `fwd_dir_8h` is its sign, a binary classification, and
`fwd_dir_8h_3c` adds a flat class for moves too small to trade - three-way, and deliberately
unbalanced, because most settlements are small. A classification request therefore resolves
more than a regression one: the class weights that correct the imbalance are fitted per fold,
because the balance of a fold is a property of its own training window and not of the panel.
It also resolves a *continuous* evaluation target, so a classifier's ranking can be scored
against the return it was trying to sign rather than against its own discrete labels.


<!-- ML4T_ZH_BEGIN c0000-p001 -->
**TabM 是集成模型，重點正是集成。** 對表格資料，平均數個獨立初始化的神經網路，是降低擬合結果不穩定性的標準做法，通常代價是必須訓練數個網路。TabM 主要只訓練一個網路。兩層骨幹由所有成員共用；每個成員只擁有一個向量，為每個隱藏單元提供一個數值，逐元素縮放骨幹輸出，並有自己的最後一層線性層。成員預測值再取平均。因此 `n_members: 4` 在 `hidden_dim: 64` 的計算成本，是一個骨幹加上四個小向量及四個輸出層，而非四個完整網路；這也解釋為何成員數可以比網路寬度提高得多。

**本 Notebook 擬合四種已宣告標籤中的三種，其中兩種不是報酬率。** `fwd_ret_8h` 是迴歸目標。`fwd_dir_8h` 是其正負號，屬二元分類；`fwd_dir_8h_3c` 則加入變動幅度小到不足以交易的平盤類別，形成三分類，且刻意不平衡，因為多數結算的變動都很小。因此分類請求比迴歸請求須多解析一項內容：修正類別不平衡的權重逐切分擬合，因為切分的類別比例是其自身訓練視窗，而非整份面板的性質。分類請求還會解析一項*連續*評估目標，使分類器排名能對照它想預測方向的原始報酬率評分，而非對照自身離散標籤。
<!-- ML4T_ZH_END c0000-p001 -->

**A neural fit has a meaningful state at every epoch**, in the way a boosted model has one at
every iteration and a linear fit does not. An epoch is one pass over the training rows. These
configurations train for 200 and save the weights every 25, so each produces eight scoreable
models rather than one and each is registered separately. What counts downstream is
configurations times checkpoints, not configurations.

**Learning objectives.** By the end of this notebook you will be able to:

- Say what a weight-sharing ensemble holds in common between its members and what it keeps
  separate, and why that makes *k* members cost far less than *k* networks.
- Tell apart a regression, a binary and a multiclass request, and say what each additionally
  resolves before anything is fitted.
- Explain why class weights are fitted per fold rather than once for the panel, and what would
  go wrong if a single weighting were carried across folds.
- Read the epoch schedule out of a declared configuration and say how many scoreable models a
  run will publish for it.
- Say why a catalog identity has to bind the device policy as well as the model and seed.

**Book reference:** Chapter 18, deep learning for tabular data.

**Prerequisites:** [`03_financial_features`](03_financial_features.ipynb) and
[`04_model_based_features`](04_model_based_features.ipynb) have written the feature matrices, and
[`05_evaluation`](05_evaluation.ipynb) has established the walk-forward folds. The canonical run
uses CUDA; the reduced run in CI does not.


<!-- ML4T_ZH_BEGIN c0000-p002 -->
**神經網路在每個訓練週期都有可評估的狀態**，如同提升模型每次迭代都有狀態，而線性模型則沒有。一個訓練週期是完整走過訓練資料列一次。這些設定訓練 200 個週期，每 25 個儲存一次權重，因此每種設定會產生八個可評分模型，而非一個，並各自登錄。下游需要計數的是設定乘以檢查點，而非只有設定。

**學習目標。** 完成本 Notebook 後，你將能夠：

- 說明共享權重的集成模型成員之間共用哪些部分、各自保留哪些部分，以及為何 *k* 個成員的成本遠低於 *k* 個完整網路。
- 區分迴歸、二元及多類別請求，說明擬合前各自還須解析什麼。
- 解釋為何類別權重逐切分擬合，而非對整份面板只擬合一次，以及跨切分沿用同一權重會出什麼問題。
- 從宣告的設定讀取訓練週期時程，說明一次執行會為該設定發布多少個可評分模型。
- 說明為何目錄識別值除了模型及隨機種子，也須包含裝置政策。

**書籍參考：**第 18 章，表格資料的深度學習。

**先備知識：**[`03_financial_features`](03_financial_features.ipynb) 與 [`04_model_based_features`](04_model_based_features.ipynb) 已寫入特徵矩陣，[`05_evaluation`](05_evaluation.ipynb) 已確立滾動向前切分。標準執行使用 CUDA；CI 中縮小規模的執行則不用。
<!-- ML4T_ZH_END c0000-p002 -->

**What it writes:** one training run per configuration and one complete validation prediction set
per checkpoint, grouped under a named population that [`13_backtest`](13_backtest.ipynb) reads
and selects from on validation backtest Sharpe. **Nothing here ranks anything**, and no number
printed below decides which model the case study goes on to use.

<!-- ML4T_ZH_BEGIN c0000-p003 -->
**寫入內容：**每種設定各有一次訓練執行，每個檢查點各有一套完整驗證預測結果，歸在具名模型群組之下；[`13_backtest`](13_backtest.ipynb) 會讀取該群組，依驗證回測夏普比率選擇。**此處不對任何結果排名**，下方印出的數值也不決定個案研究後續使用哪個模型。
<!-- ML4T_ZH_END c0000-p003 -->


In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import (
    ALL_LABELS,
    OFFICIAL_POPULATION,
    declared_contracts,
    freeze_official_model_population,
    model_request_catalog,
    open_study,
    plan_model_catalog,
    plan_specs,
    run_model_plan,
)
from case_studies.research import SUPERSEDES_LIVE, supersedes_for_run

In [2]:
EXECUTION_TIER = "canonical"
# That this run intends to extend the lineage published under the case-wide official model
# population's name. `SUPERSEDES_LIVE` says so without quoting a hash, which would have to equal
# a value the registry moves on every publish; `supersedes_for_run` resolves it against the
# registry and withholds it on a clean clone or under a preview.
SUPERSEDES_POPULATION: str = SUPERSEDES_LIVE
# The same declaration for this notebook's own checkpoint population, which is a separate
# lineage: a refit can move either without moving the other.
SUPERSEDES_MODEL_POPULATION: str = SUPERSEDES_LIVE
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABELS = ALL_LABELS
PREVIEW_REDUCTIONS = {}
OVERRIDES = {"class_weight": "balanced", "device": "cuda"}

## 1. Resolve targets, imbalance policy, and checkpoints

Nothing is fitted below. The catalog resolves each declared configuration against each label into
a request with an identity, and the table that follows prints what those requests will actually
do. Three fields on it repay attention.

`task` is where the three labels stop being interchangeable. A regression request minimizes
squared error against `fwd_ret_8h`; a binary request fits the sign; the three-class request fits
a sign with a flat band in the middle. They are different objectives on the same features, and a
comparison across them is a comparison of what each was asked to do, not of which is better.

`class_weights` is empty for the regression request and populated for the other two, and it is
resolved **per fold**. The proportion of flat settlements is a property of a particular training
window, not of the panel: crypto funding regimes are long-lived, and a fold covering a quiet
stretch has a different balance from one covering a volatile stretch. A single weighting computed
once over the whole panel would carry each fold a correction fitted partly on the others.

`checkpoint_schedule` is what turns each configuration into several scoreable models. Read the
epoch count and interval off this table rather than from the configuration file, because it is
the frozen specification that the run will follow.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 1. 解析目標、類別不平衡政策與檢查點

下方不擬合任何模型。目錄會將每項已宣告設定與每種標籤結合，解析成有識別值的請求；後續表格列出請求實際會做什麼。其中三個欄位值得注意。

`task` 顯示三種標籤從何處開始有不同含義。迴歸請求對 `fwd_ret_8h` 最小化平方誤差；二元請求擬合正負號；三分類請求擬合帶有中間平盤區間的正負方向。它們使用相同特徵，但目標函數不同；跨目標比較應比較各自被要求完成的任務，而非單純判定誰較好。

`class_weights` 對迴歸請求為空，對其他兩種則有值，而且是**逐切分**解析。平盤結算占比取決於特定訓練視窗，而非整份面板；加密貨幣資金費率市場狀態持續較久，涵蓋平靜區段的切分，與涵蓋劇烈波動區段的切分，類別比例不同。若對整份面板只計算一組權重，各切分的修正值就會部分根據其他切分資料擬合。

`checkpoint_schedule` 使每種設定變成數個可評分模型。應從此表而非設定檔讀取訓練週期數及間隔，因為這才是執行會遵循的已固定規格。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
official_population = (
    freeze_official_model_population(
        study,
        supersedes=supersedes_for_run(
            study,
            population_name=OFFICIAL_POPULATION,
            declared=SUPERSEDES_POPULATION or None,
            execution_tier=EXECUTION_TIER,
        ),
    )
    if EXECUTION_TIER == "canonical"
    else None
)
requests = model_request_catalog("tabular_dl", labels=LABELS, config_prefix="tabm")
requests

family,label,config_name
str,str,str
"""tabular_dl""","""fwd_ret_8h""","""tabm_s"""
"""tabular_dl""","""fwd_ret_8h""","""tabm_m"""
"""tabular_dl""","""fwd_ret_8h""","""tabm_l"""
"""tabular_dl""","""fwd_ret_24h""","""tabm_s"""
"""tabular_dl""","""fwd_ret_24h""","""tabm_m"""
…,…,…
"""tabular_dl""","""fwd_dir_8h""","""tabm_m"""
"""tabular_dl""","""fwd_dir_8h""","""tabm_l"""
"""tabular_dl""","""fwd_dir_8h_3c""","""tabm_s"""


In [4]:
plan = plan_model_catalog(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides=OVERRIDES,
    preview_reductions=PREVIEW_REDUCTIONS,
)
# Task semantics and imbalance treatment are resolved inputs, so read them from the frozen
# specification rather than restating the configuration file here.
resolved_tasks = [spec["computation"]["task"] for spec in plan_specs(plan)]
resolved_contracts = declared_contracts(plan).with_columns(
    pl.Series("metrics", [task.get("metrics", []) for task in resolved_tasks]),
    pl.Series("imbalance", [task.get("imbalance") for task in resolved_tasks]),
)
resolved_contracts.select(
    "label",
    "config_name",
    "task",
    "continuous_eval_label",
    "imbalance",
    "metrics",
    "checkpoint_value",
    "eligible_rows",
    "training_hash",
)

label,config_name,task,continuous_eval_label,imbalance,metrics,checkpoint_value,eligible_rows,training_hash
str,str,str,str,struct[2],list[str],i64,i64,str
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],25,35280,"""e15ee733b2e3"""
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],50,35280,"""e15ee733b2e3"""
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],75,35280,"""e15ee733b2e3"""
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],100,35280,"""e15ee733b2e3"""
"""fwd_ret_8h""","""tabm_s""","""regression""",null,null,[],125,35280,"""e15ee733b2e3"""
…,…,…,…,…,…,…,…,…
"""fwd_dir_8h_3c""","""tabm_l""","""classification""","""fwd_ret_8h""","{{[0.921834, 1.271327, 0.886033],[0.902901, 1.210376, 0.937849]},""balanced""}","[""ic"", ""accuracy"", ""balanced_accuracy""]",100,35280,"""3b6cafb60fbd"""
"""fwd_dir_8h_3c""","""tabm_l""","""classification""","""fwd_ret_8h""","{{[0.921834, 1.271327, 0.886033],[0.902901, 1.210376, 0.937849]},""balanced""}","[""ic"", ""accuracy"", ""balanced_accuracy""]",125,35280,"""3b6cafb60fbd"""
"""fwd_dir_8h_3c""","""tabm_l""","""classification""","""fwd_ret_8h""","{{[0.921834, 1.271327, 0.886033],[0.902901, 1.210376, 0.937849]},""balanced""}","[""ic"", ""accuracy"", ""balanced_accuracy""]",150,35280,"""3b6cafb60fbd"""


The complete case-wide population is recorded before the first fit, so a member that later
fails to train cannot quietly disappear from the population it was declared in. This notebook
produces one slice of it, and that slice must lie inside the declaration.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
完整的個案研究模型群組在首次擬合前就已記錄，因此後續訓練失敗的成員，不能默默從已宣告群組中消失。本 Notebook 產生其中一部分，而且必須包含在完整宣告中。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
if official_population is not None:
    outside = set(plan.expected_prediction_hashes) - set(official_population.members)
    if outside:
        raise RuntimeError(
            f"{len(outside)} declared checkpoints lie outside the official model population"
        )

## 2. Execute and validate the fitted-state population

Each configuration is fitted on each fold; the weights are persisted at every checkpoint epoch
with a digest, and one complete validation prediction set is registered per checkpoint. A cached
fitted state is reused only when its digest matches, so a resumed run cannot silently continue
from weights that a code change has invalidated.

The completeness check is the substantive one. A prediction set is complete when it covers every
validation key its fold declares. A set covering most of them is not a slightly worse result - it
is a different sample, and putting it beside a complete one in the backtest would compare two
models measured on different data. The run raises rather than publishing an incomplete
population, which is the behaviour to want: a loud failure here costs a re-run, and a quiet one
costs a wrong comparison that nothing downstream can detect.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 執行並驗證已擬合狀態的模型群組

每種設定都在每個切分上擬合；每個檢查點週期都將權重及摘要持久化，並為每個檢查點登錄一套完整驗證預測結果。只有已快取擬合狀態的摘要相符時，才會重用，因此接續執行時，不會默默沿用已因程式碼變更而失效的權重。

完整性檢查是關鍵。若預測集涵蓋切分宣告的每個驗證鍵值，才算完整。只涵蓋大部分鍵值的預測集，不是結果稍差，而是使用不同樣本；若將它與完整預測集並列回測，就等於比較在不同資料上衡量的兩個模型。遇到不完整群組時，執行會報錯，而非發布；這是正確做法：此處明顯失敗只需重新執行，默默失敗則會導致錯誤比較，且下游無法察覺。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
execution = run_model_plan(
    plan,
    supersedes=supersedes_for_run(
        study,
        population_name="crypto-tabm-validation-predictions-v1",
        declared=SUPERSEDES_MODEL_POPULATION or None,
        execution_tier=EXECUTION_TIER,
    ),
    population_name="crypto-tabm-validation-predictions-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if (
    catalog.height != len(plan.expected_prediction_hashes)
    or catalog.filter(~pl.col("complete")).height
):
    raise RuntimeError("TabM fitted-state or prediction population is incomplete")
catalog.select(
    "label",
    "config_name",
    "task",
    "checkpoint_kind",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
    "complete",
)

Preparing and releasing folds...
  Fold 0: train=23,681  val=16,738


      epoch  25/200: loss=0.001901, IC=-0.0005


      epoch  50/200: loss=0.001790, IC=+0.0127


      epoch  75/200: loss=0.001767, IC=+0.0179


      epoch 100/200: loss=0.001760, IC=+0.0188


      epoch 125/200: loss=0.001740, IC=+0.0209


... (231 outputs left out)

    → best_epoch=75, IC=+0.0328 (15.5s)


    → best_epoch=25, IC=+0.0313 (32.9s)



  Best: cd943aa613be @ epoch 75 (IC=+0.0328)


label,config_name,task,checkpoint_kind,checkpoint_value,training_hash,prediction_hash,complete
str,str,str,str,i64,str,str,bool
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",25,"""cf8e1d374c66""","""a72bb59da07e""",true
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",50,"""cf8e1d374c66""","""e09b295d3816""",true
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",75,"""cf8e1d374c66""","""4d864353c7dc""",true
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",100,"""cf8e1d374c66""","""199b71b6f6eb""",true
"""fwd_dir_8h""","""tabm_l""","""classification""","""epoch""",125,"""cf8e1d374c66""","""b65193bda646""",true
…,…,…,…,…,…,…,…
"""fwd_ret_8h""","""tabm_s""","""regression""","""epoch""",100,"""e15ee733b2e3""","""570438b543a1""",true
"""fwd_ret_8h""","""tabm_s""","""regression""","""epoch""",125,"""e15ee733b2e3""","""402b0307a5f5""",true
"""fwd_ret_8h""","""tabm_s""","""regression""","""epoch""",150,"""e15ee733b2e3""","""592bd78868aa""",true


## Key takeaways and limitations


<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 重點摘要與限制
<!-- ML4T_ZH_END c0010-p000 -->

- **The ensemble is nearly free, and that is the design.** Four members share one two-layer
  backbone and own only a per-unit scaling vector and a final linear layer each. The averaging
  that steadies a neural fit on a table costs four small tensors here rather than four networks,
  which is why the member count is the cheap dial and the hidden width is not.
- **Task semantics and imbalance treatment are resolved inputs, not notebook conventions.** What
  objective is minimized, and how a fold's class imbalance is corrected, are read back out of the
  frozen specification. If they were decided in notebook code, two runs of the same declared
  configuration could differ without their identities differing.
- **Class weights belong to a fold, not to the panel.** Fitting them once over the whole history
  would carry every fold a correction estimated partly on windows it must not see.
- **Configurations times checkpoints is the count that matters.** Eight scoreable models per
  configuration, each registered separately. Reporting the best of them as a single model's score
  would be reporting a maximum over eight draws, and the selection that handles this correctly
  happens in [`13_backtest`](13_backtest.ipynb), not here.
- **The identity binds the device policy, not only the model and seed.** GPU kernels reorder
  floating-point reductions, so the same weights and the same data can produce slightly different
  numbers on a different device. Binding the device policy into the identity means a result is
  never compared against one produced under a different arithmetic.
- **Two folds is the binding constraint, not the architecture.** As with every model family in
  this case study, the usable perpetual funding history is short, and no amount of capacity
  compensates for that.

<!-- ML4T_ZH_BEGIN c0010-p001 -->
- **集成成本幾乎可忽略，這正是設計目的。** 四個成員共用一個兩層骨幹，各自只有逐單元縮放向量與最後一層線性層。讓表格神經網路擬合結果更穩定的平均運算，在此只需四組小型張量，而非四個完整網路，因此調整成員數量的成本低，增加隱藏層寬度則不然。
- **任務含義與不平衡處理是解析後的輸入，而非 Notebook 的慣例。** 最小化哪種目標函數，以及如何修正各切分的類別不平衡，都從已固定規格讀回。若在 Notebook 程式碼中決定，兩次宣告設定相同的執行，可能產生不同結果，識別值卻相同。
- **類別權重屬於切分，而非整份面板。** 若在整段歷史資料上只擬合一次，每個切分使用的修正值都會部分根據它不應看見的視窗估計。
- **真正重要的計數是設定乘以檢查點。** 每種設定有八個可評分模型，各自登錄。若將其中最佳者報成單一模型的評分，就等於將八次測量的最大值當成一次結果；正確處理這項選擇的是 [`13_backtest`](13_backtest.ipynb)，不是此處。
- **識別值包含裝置政策，而非只有模型與隨機種子。** GPU 核心會改變浮點數歸約的順序，因此即使用相同權重及資料，在不同裝置上仍可能產生略有差異的數字。將裝置政策納入識別值，確保結果不會與使用不同運算方式產生者直接比較。
- **真正的限制是只有兩個切分，而非架構。** 如同本個案研究的其他模型類別，可用的永續合約資金費率歷史資料較短，再高的模型容量也無法補救。
<!-- ML4T_ZH_END c0010-p001 -->
